In [0]:
files = dbutils.fs.ls("/Volumes/gr5069/raw/f1_data/")
print(len(files))
for f in files:
    print(f.name)

# Step 1: Create Two Tables for Predictions 

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS gr5069.kc3949.predictions_logreg (
    raceId INT,
    driverId INT,
    constructorId INT,
    grid INT,
    actual_top10 INT,
    predicted_top10 INT,
    prediction_probability DOUBLE,
    model_name STRING,
    run_id STRING,
    prediction_timestamp TIMESTAMP
)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS gr5069.kc3949.predictions_rf (
    raceId INT,
    driverId INT,
    constructorId INT,
    grid INT,
    actual_top10 INT,
    predicted_top10 INT,
    prediction_probability DOUBLE,
    model_name STRING,
    run_id STRING,
    prediction_timestamp TIMESTAMP
)
""")

print("Tables created successfully")

# Step 2: Data Preparation

In [0]:
# Load and merge F1 data
import pandas as pd

base_path = "/Volumes/gr5069/raw/f1_data/"

# Load core tables
results = pd.read_csv(base_path + "results.csv")
races = pd.read_csv(base_path + "races.csv")
drivers = pd.read_csv(base_path + "drivers.csv")
constructors = pd.read_csv(base_path + "constructors.csv")

# Replace '\N' (F1 dataset's missing value marker) with proper NaN
results = results.replace(r'\N', pd.NA)

# Cast numeric columns
results['position'] = pd.to_numeric(results['position'], errors='coerce')
results['grid'] = pd.to_numeric(results['grid'], errors='coerce')

# Build modeling dataframe: predict whether driver finishes in top 10
df = results[['raceId', 'driverId', 'constructorId', 'grid', 'position', 'statusId']].copy()
df = df.dropna(subset=['position', 'grid'])
df['top10'] = (df['position'] <= 10).astype(int)  # binary target

print("Dataset shape:", df.shape)
print("Class balance:\n", df['top10'].value_counts(normalize=True))
df.head()

In [0]:
# Train/test split
from sklearn.model_selection import train_test_split

features = ['grid', 'constructorId', 'driverId', 'raceId']
X = df[features]
y = df['top10']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Keep identifiers for the test set so we can write predictions back to the database
test_ids = df.loc[X_test.index, ['raceId', 'driverId', 'constructorId', 'grid', 'top10']].copy()
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

# Step 3: Build Two MLflow Models

In [0]:
# Cell: HW5 - Two models with full MLflow logging
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from mlflow.models.signature import infer_signature

# Reuse the experiment that already works
mlflow.set_experiment("/Users/kc3949@columbia.edu/f1_model")

FEATURES = features  # from the data prep cell above

# Make sure no run is lingering from previous failures
try:
    mlflow.end_run()
except:
    pass

# Model 1: Logistic Regression
with mlflow.start_run(run_name="hw5_logreg") as run:
    rid_lr = run.info.run_id

    # Hyperparameters
    params_lr = {"C": 1.0, "max_iter": 1000, "solver": "lbfgs", "penalty": "l2"}
    mlflow.log_params(params_lr)

    # Train (scaling helps logistic regression)
    model_lr = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(**params_lr, random_state=42))
    ])
    model_lr.fit(X_train, y_train)

    y_pred_lr = model_lr.predict(X_test)
    y_prob_lr = model_lr.predict_proba(X_test)[:, 1]

    # Four metrics
    metrics_lr = {
        "accuracy":  accuracy_score(y_test, y_pred_lr),
        "precision": precision_score(y_test, y_pred_lr, zero_division=0),
        "recall":    recall_score(y_test, y_pred_lr, zero_division=0),
        "f1":        f1_score(y_test, y_pred_lr, zero_division=0),
        "roc_auc":   roc_auc_score(y_test, y_prob_lr),
    }
    mlflow.log_metrics(metrics_lr)

    # Log the model itself
    mlflow.sklearn.log_model(
        model_lr, "model",
        signature=infer_signature(X_train, y_pred_lr),
        input_example=X_train.head(3),
    )

    # Artifact 1: Confusion Matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        confusion_matrix(y_test, y_pred_lr),
        annot=True, fmt="d",
        xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=ax
    )
    ax.set_title("Confusion Matrix - Logistic Regression")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.tight_layout()
    fig.savefig("/tmp/cm_lr.png")
    plt.close()
    mlflow.log_artifact("/tmp/cm_lr.png", "plots")

    # Artifact 2: Predictions CSV
    pd.DataFrame({
        "actual": y_test.values,
        "predicted": y_pred_lr,
        "prob": y_prob_lr,
    }).to_csv("/tmp/preds_lr.csv", index=False)
    mlflow.log_artifact("/tmp/preds_lr.csv", "csv")

    print(f"Logistic Regression done | run_id={rid_lr}")
    print(f"  Accuracy={metrics_lr['accuracy']:.3f}  F1={metrics_lr['f1']:.3f}  AUC={metrics_lr['roc_auc']:.3f}")

# Model 2: Random Forest

In [0]:

# Model 2: Random Forest
with mlflow.start_run(run_name="hw5_rf") as run:
    rid_rf = run.info.run_id

    # Hyperparameters
    params_rf = {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
        "class_weight": "balanced",
    }
    mlflow.log_params(params_rf)

    # Train
    model_rf = RandomForestClassifier(**params_rf, random_state=42, n_jobs=-1)
    model_rf.fit(X_train, y_train)

    y_pred_rf = model_rf.predict(X_test)
    y_prob_rf = model_rf.predict_proba(X_test)[:, 1]

    # Four metrics
    metrics_rf = {
        "accuracy":  accuracy_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf, zero_division=0),
        "recall":    recall_score(y_test, y_pred_rf, zero_division=0),
        "f1":        f1_score(y_test, y_pred_rf, zero_division=0),
        "roc_auc":   roc_auc_score(y_test, y_prob_rf),
    }
    mlflow.log_metrics(metrics_rf)

    # Log the model itself
    mlflow.sklearn.log_model(
        model_rf, "model",
        signature=infer_signature(X_train, y_pred_rf),
        input_example=X_train.head(3),
    )

    # Artifact 1: Confusion Matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        confusion_matrix(y_test, y_pred_rf),
        annot=True, fmt="d",
        xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=ax
    )
    ax.set_title("Confusion Matrix - Random Forest")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.tight_layout()
    fig.savefig("/tmp/cm_rf.png")
    plt.close()
    mlflow.log_artifact("/tmp/cm_rf.png", "plots")

    # Artifact 2: Feature Importance
    imp = pd.Series(model_rf.feature_importances_, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(6, 4))
    imp.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Feature Importances - Random Forest")
    fig.tight_layout()
    fig.savefig("/tmp/fi_rf.png")
    plt.close()
    mlflow.log_artifact("/tmp/fi_rf.png", "plots")

    print(f"Random Forest done | run_id={rid_rf}")
    print(f"  Accuracy={metrics_rf['accuracy']:.3f}  F1={metrics_rf['f1']:.3f}  AUC={metrics_rf['roc_auc']:.3f}")

print(f"\nBoth models logged. Run IDs:\n  LR: {rid_lr}\n  RF: {rid_rf}")